# Chapter 07 · Notebook 01: Freeze weights and train the bias

**Optional math reference:** [Mathematical view](00_math_intuition.ipynb), available for later. These programming lessons can be followed independently.

[Chapter guide](README.md) · [Previous chapter](../06_datasets_and_dataloaders/README.md) · [Course index](../README.md)

Until now we have trained all the parameters in our small model. Now choose which parameters can learn. Start by freezing the two weights while keeping the bias trainable.

This is a foundation for fine-tuning: some existing parameter values can be kept fixed while selected values are trained. We use our familiar regression model so we can inspect the effect directly.

Start in a fresh kernel. This notebook defines its own model and data and does not need an earlier notebook or saved checkpoint. Run one lesson at a time. To repeat the original experiment after later training cells, rerun Setup first.

## Contents

1. [Freeze the weights and inspect gradients](#freeze-gradients)
2. [Update only the bias](#bias-update)
3. [Repeat bias-only training with fresh gradients](#bias-only-training)
4. [Unfreeze weights and include them in the optimizer](#unfreeze-weights)
5. [Continue training all parameters](#all-parameter-training)

## Setup: recreate our familiar model and batch

Our `Linear(2, 1)` layer contains two weights and one bias. Initialize weights to `[1, 1]` and bias to `0`. The two inputs are `[1, 0]` and `[0, 1]`, with correct targets `5` and `3`.


In [ ]:
import torch

# Start with a fresh layer; its parameters are trainable by default.
model = torch.nn.Linear(2, 1)
with torch.no_grad():
    model.weight.fill_(1.)
    model.bias.fill_(0.)

x = torch.tensor([[1., 0.], [0., 1.]])
y = torch.tensor([[5.], [3.]])
loss_fn = torch.nn.MSELoss()

print("Initial weights:", model.weight.detach().flatten().tolist())
print("Initial bias:", model.bias.item())
print("Weight requires gradients:", model.weight.requires_grad)
print("Bias requires gradients:", model.bias.requires_grad)


<a id="freeze-gradients"></a>
## Lesson 01: freeze the weights and inspect gradients

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#constraint).

**Purpose:** keep the weights out of gradient calculation while still calculating a bias gradient.

`model.weight.requires_grad_(False)` changes the existing weight tensor's gradient flag. That tensor contains both weights, so both are frozen. The weights still participate in prediction; their values are not removed or replaced. The bias keeps `requires_grad=True`.

Keep the forward calculation and loss outside `torch.no_grad()` because we need a tracked calculation for the trainable bias. Freezing selected parameters differs from disabling gradient recording for the entire forward calculation.

Clear stored gradients before this fresh calculation. Freezing a parameter does not itself clear a previously stored gradient; our fresh model starts without any, and `model.zero_grad(set_to_none=True)` also removes any stored gradients if the cell is repeated.

No optimizer step is performed in this lesson. `backward()` calculates the bias gradient while leaving all parameter values unchanged. In the next lesson we will register the bias with an optimizer and update it.

**Before running:** Should the weight gradient be a tensor or `None`? Which parameter should receive a gradient? Should any parameter values change after `backward()`?

**My prediction and reason:** _write here_


In [ ]:
# Disable gradient calculation for the tensor containing both weights.
model.weight.requires_grad_(False)

# Clear previous stored gradients without changing parameter values or flags.
model.zero_grad(set_to_none=True)

# Keep this calculation tracked so the trainable bias can receive a gradient.
predictions = model(x)
loss = loss_fn(predictions, y)
loss.backward()  # Calculate gradients only; no parameter update yet.

print("Weight requires gradients:", model.weight.requires_grad)
print("Bias requires gradients:", model.bias.requires_grad)
print("Predictions:", predictions.detach().flatten().tolist())
print("Loss:", loss.item())
print("Weight gradient:", model.weight.grad)
print("Bias gradient:", model.bias.grad.item())
print("Weights after backward:", model.weight.detach().flatten().tolist())
print("Bias after backward:", model.bias.item())

# Count scalar parameter values, including both values inside the weight tensor.
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
print("Total parameter values:", total_parameters)
print("Trainable parameter values:", trainable_parameters)


**My observation and explanation:** _write here_

The total parameter count includes frozen values. Freezing changes which parameters receive gradients; it does not change the layer's prediction formula or remove its weights.

`None` means no weight gradient is stored in this experiment. It is different from a computed gradient whose numerical value is zero.



<a id="bias-update"></a>
## Lesson 02: update only the bias

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#constraint).

**Purpose:** let SGD update the bias while leaving both frozen weights unchanged.

Run this cell once immediately after Lesson 01. That lesson already calculated and stored the bias gradient, `-6`, at bias `0`. We use that stored gradient for one update with learning rate `0.05`: `0 - 0.05 * (-6) = 0.3`.

Create an optimizer containing only `model.bias`. The weight tensor is frozen, and it is also absent from this optimizer's parameter list. SGD's `step()` applies the existing gradient; it does not calculate a new gradient.

We do not clear gradients between Lesson 01's `backward()` and this `step()`, because the step needs that calculated gradient. Before another training update, clear the old gradients and recalculate from the current model values.

After the step, measure predictions and loss without recording a new graph. Both inputs had prediction `1` before training, so changing only the shared bias to `0.3` should move both predictions to `1.3`.

Rerunning this update cell alone would reuse the old stored gradient and apply another update. To reproduce the original one-step experiment, rerun Setup, Lesson 01, then this cell.

**Before running:** Predict the new bias, both weight values, and the updated predictions.

**My prediction and reason:** _write here_


In [ ]:
# Register only the bias; the optimizer does not contain the weight tensor.
optimizer = torch.optim.SGD([model.bias], lr=0.05)

# Keep an independent copy of the weights to compare after the update.
weights_before_step = model.weight.detach().clone()
print("Weights before step:", weights_before_step.flatten().tolist())
print("Bias before step:", model.bias.item())
print("Stored bias gradient:", model.bias.grad.item())
print("Loss before step:", loss.item())

# Use the bias gradient calculated by backward() in Lesson 01.
optimizer.step()

# Measure the effect of this update without calculating new gradients.
with torch.no_grad():
    updated_predictions = model(x)
    updated_loss = loss_fn(updated_predictions, y)

print("Weights after step:", model.weight.detach().flatten().tolist())
print("Bias after step:", model.bias.item())
print("Updated predictions:", updated_predictions.flatten().tolist())
print("Updated loss:", updated_loss.item())
print("Weights unchanged:", torch.equal(model.weight.detach(), weights_before_step))
print("Weight gradient still:", model.weight.grad)


**My observation and explanation:** _write here_

Only the bias changed. The frozen weights still contribute their values to predictions, and the shared bias shifts both predictions by the same amount. Loss decreased even though two of the model's three parameter values stayed fixed.

The numerical bias may print as `0.30000001192092896` because this tensor uses floating-point arithmetic. It represents our expected update to approximately `0.3`.



<a id="bias-only-training"></a>
## Lesson 03: repeat bias-only training with fresh gradients

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#loss-floor).

**Purpose:** observe the limit of training only a shared bias when frozen weights make two examples have the same prediction.

For inputs `[1, 0]` and `[0, 1]`, frozen weights `[1, 1]` give the same weighted-input value, `1`. Both predictions are therefore `1 + bias`. Their targets are different: `5` and `3`. One shared bias cannot make both exactly correct at the same time.

Start a separate, repeatable experiment with a new model at weights `[1, 1]` and bias `0`. Freeze its weights and register only its bias with a fresh SGD optimizer. This experiment leaves the earlier `model` and `optimizer` untouched.

Train on both examples together for 100 updates. Each iteration clears the previous bias gradient, calculates a new loss at the current bias, calls `backward()`, and applies one update. Each printed loss is measured on both examples after that update.

The optimizer minimizes their mean squared loss together. Reducing the loss is possible even when the chosen trainable parameters cannot fit all targets exactly.

**Before running:** Can the final loss approach zero with these fixed weights? Do you expect the two predictions to become different from one another?

**My prediction and reason:** _write here_


In [ ]:
# Start a fresh bias-only experiment so rerunning this cell is repeatable.
bias_only_model = torch.nn.Linear(2, 1)
with torch.no_grad():
    bias_only_model.weight.fill_(1.)
    bias_only_model.bias.fill_(0.)
bias_only_model.weight.requires_grad_(False)

bias_only_optimizer = torch.optim.SGD([bias_only_model.bias], lr=0.05)
bias_only_model.train()
number_of_updates = 100
report_updates = {1, 2, 5, 10, 20, 50, 100}

with torch.no_grad():
    initial_bias_only_loss = loss_fn(bias_only_model(x), y).item()
print("Initial loss:", initial_bias_only_loss)

# Each update uses both examples and a freshly calculated bias gradient.
for update in range(1, number_of_updates + 1):
    bias_only_optimizer.zero_grad(set_to_none=True)
    bias_only_predictions = bias_only_model(x)
    bias_only_loss = loss_fn(bias_only_predictions, y)
    bias_only_loss.backward()
    bias_only_optimizer.step()

    if update in report_updates:
        with torch.no_grad():
            loss_after_update = loss_fn(bias_only_model(x), y).item()
        print(
            f"Update {update:3d} | bias={bias_only_model.bias.item():.6f} | "
            f"loss={loss_after_update:.6f}"
        )

# Inspect predictions and each example's remaining error after training.
with torch.no_grad():
    final_bias_predictions = bias_only_model(x)
    final_squared_errors = (final_bias_predictions - y) ** 2
    final_bias_only_loss = loss_fn(final_bias_predictions, y)

print("Final weights:", bias_only_model.weight.flatten().tolist())
print("Final bias:", bias_only_model.bias.item())
print("Final predictions:", final_bias_predictions.flatten().tolist())
print("Targets:", y.flatten().tolist())
print("Squared errors:", final_squared_errors.flatten().tolist())
print("Final mean squared loss:", final_bias_only_loss.item())
print("Weight gradient:", bias_only_model.weight.grad)


**My observation and explanation:** _write here_

Explain the remaining loss using the equal predictions and different targets. Freezing reduces the set of parameter values the optimizer can change. Whether the model can fit the data depends on those fixed values and which parameters remain trainable; bias-only training does not always have a positive minimum loss for every possible dataset.



<a id="unfreeze-weights"></a>
## Lesson 04: unfreeze weights and include them in the optimizer

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#constraint).

**Purpose:** distinguish enabling gradients from registering parameters with the optimizer.

Enabling `requires_grad` lets a parameter receive gradients. An optimizer updates only the parameters registered with that optimizer. Changing a gradient flag does not add that parameter to an existing optimizer's list.

Use a fresh demonstration model at weights `[1, 1]` and bias `3`, the ideal bias-only solution from Lesson 03. Its predictions are both `4`, with mean squared loss `1`. This exact starting bias keeps the comparison easy to interpret; the earlier bias-only experiment remains unchanged.

First freeze the weights and create a bias-only optimizer. Then enable weight gradients with `requires_grad_(True)`. Calculate gradients and call that existing optimizer's `step()`. We expect weight gradients, but the weights should stay fixed because the optimizer still contains only the bias. The bias gradient is zero at this starting point.

Next create a new plain SGD optimizer containing `unfreeze_model.parameters()`. Clear gradients and recalculate at the current values before its step. Now both weights can be updated. Recreating the optimizer is appropriate for this plain-SGD demonstration; it does not preserve optimizer history from an existing momentum run.

This cell creates its own demonstration model, so rerunning it repeats the same comparison. It needs Setup's `x`, `y`, and `loss_fn`.

**Before running:** Which phase should change the weights? Can changing the weights let the two predictions move in different directions?

**My prediction and reason:** _write here_


In [ ]:
# Recreate the exact best bias-only solution in a separate model.
unfreeze_model = torch.nn.Linear(2, 1)
with torch.no_grad():
    unfreeze_model.weight.fill_(1.)
    unfreeze_model.bias.fill_(3.)
unfreeze_model.weight.requires_grad_(False)

# This optimizer initially contains only the bias.
unfreeze_optimizer = torch.optim.SGD([unfreeze_model.bias], lr=0.05)

# Phase A: enable weight gradients without changing the optimizer's parameter list.
unfreeze_model.weight.requires_grad_(True)
unfreeze_model.zero_grad(set_to_none=True)  # Clear gradients for all model parameters.
phase_a_loss = loss_fn(unfreeze_model(x), y)
phase_a_loss.backward()
print("Phase A: unfrozen weights, bias-only optimizer")
print("  Weight gradients:", unfreeze_model.weight.grad.flatten().tolist())
print("  Bias gradient:", unfreeze_model.bias.grad.item())
unfreeze_optimizer.step()

with torch.no_grad():
    phase_a_predictions = unfreeze_model(x)
    phase_a_after_loss = loss_fn(phase_a_predictions, y)
print("  Weights after step:", unfreeze_model.weight.detach().flatten().tolist())
print("  Bias after step:", unfreeze_model.bias.item())
print("  Predictions:", phase_a_predictions.flatten().tolist())
print("  Loss:", phase_a_after_loss.item())

# Phase B: register both the weight tensor and the bias with a new optimizer.
unfreeze_optimizer = torch.optim.SGD(unfreeze_model.parameters(), lr=0.05)
unfreeze_optimizer.zero_grad(set_to_none=True)
phase_b_loss = loss_fn(unfreeze_model(x), y)
phase_b_loss.backward()  # Calculate fresh gradients before the next step.
print("Phase B: unfrozen weights, optimizer contains all parameters")
print("  Weight gradients:", unfreeze_model.weight.grad.flatten().tolist())
print("  Bias gradient:", unfreeze_model.bias.grad.item())
unfreeze_optimizer.step()

with torch.no_grad():
    phase_b_predictions = unfreeze_model(x)
    phase_b_after_loss = loss_fn(phase_b_predictions, y)
print("  Weights after step:", unfreeze_model.weight.detach().flatten().tolist())
print("  Bias after step:", unfreeze_model.bias.item())
print("  Predictions:", phase_b_predictions.flatten().tolist())
print("  Loss:", phase_b_after_loss.item())


**My observation and explanation:** _write here_

Both phases calculated weight gradients. Only Phase B's optimizer included the weights, so only that phase changed their values. The opposite signs of the weight gradients allow the two predictions to move toward their different targets.

The zero bias gradient here means the current shared prediction is already the best compromise while weights are fixed. It does not mean all model parameters are optimal once the weights are allowed to change.



<a id="all-parameter-training"></a>
## Lesson 05: continue training all parameters

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#unfreeze).

**Purpose:** see how allowing both weights to learn removes the equal-prediction restriction on these two examples.

Continue from Lesson 04's Phase B model and optimizer. Both weights require gradients, and the optimizer contains both the weight tensor and the bias. Phase B already performed one step; this cell performs 100 additional updates on the same two examples.

At each update, clear the previous gradients, calculate predictions and mean squared loss, call `backward()`, and call `step()`. Evaluate the current loss afterward without a new gradient calculation.

The bias-only experiment could not make its equal predictions match targets `5` and `3` together. The unfrozen weights can move those predictions independently. For example, weights `[2, 0]` and bias `3` would fit these two examples exactly. Finite training can approach such a solution without producing an exactly zero printed loss.

To repeat this continuation from the same starting point, rerun Lesson 04, then this cell. It uses that lesson's separate `unfreeze_model` and does not modify the earlier bias-only experiment.

**Before running:** Can the loss go below the bias-only minimum of `1`? Must every trainable parameter change on every step?

**My prediction and reason:** _write here_


In [ ]:
# Continue the all-parameter optimizer created in Lesson 04, Phase B.
unfreeze_model.train()
additional_updates = 100
report_updates = {1, 2, 5, 10, 20, 50, 100}

with torch.no_grad():
    loss_before_continuing = loss_fn(unfreeze_model(x), y).item()
print("Loss before continuing:", loss_before_continuing)

# Use fresh gradients at the current parameter values for every update.
for additional_update in range(1, additional_updates + 1):
    unfreeze_optimizer.zero_grad(set_to_none=True)
    all_parameter_predictions = unfreeze_model(x)
    all_parameter_loss = loss_fn(all_parameter_predictions, y)
    all_parameter_loss.backward()
    unfreeze_optimizer.step()

    if additional_update in report_updates:
        with torch.no_grad():
            current_loss = loss_fn(unfreeze_model(x), y).item()
        print(f"Additional update {additional_update:3d} | loss={current_loss:.8f}")

# Inspect the predictions on the same two training examples.
with torch.no_grad():
    final_all_predictions = unfreeze_model(x)
    final_all_loss = loss_fn(final_all_predictions, y)

print("Additional updates completed:", additional_updates)
print("Final weights:", unfreeze_model.weight.detach().flatten().tolist())
print("Final bias:", unfreeze_model.bias.item())
print("Final predictions:", final_all_predictions.flatten().tolist())
print("Targets:", y.flatten().tolist())
print("Final mean squared loss:", final_all_loss.item())
print("Best possible loss with weights fixed at [1, 1]:", 1.0)


**My observation and explanation:** _write here_

The loss can now fall below `1` because the predictions no longer have to be equal. The bias may remain nearly unchanged in this symmetric experiment while the weights learn.

These two examples do not uniquely determine three parameter values. Both weights `[2, 0]` with bias `3` and weights `[3, 1]` with bias `2` predict the correct targets for these inputs. Fitting these two training examples does not prove we recovered the original rule or that predictions on new inputs will be correct.

<a id="chapter-07-review"></a>
## Chapter 07 review

All five lessons were run and reviewed in the mentoring conversation.

- Frozen weights still contribute to predictions. Setting `requires_grad_(False)` prevents new gradients from being accumulated for them; it does not remove their values or clear an already stored gradient.
- Keep training calculations tracked for the parameters that remain trainable. Wrapping the whole forward calculation in `no_grad()` would prevent the bias gradient needed for this experiment.
- Gradient calculation and optimizer registration are separate choices. Enabling weight gradients does not add the weights to an existing bias-only optimizer.
- A fresh parameter with no gradient stored has `.grad is None`. A computed gradient with value zero is a different case.
- `backward()` calculates gradients without updating parameter values. `step()` uses the gradients to update parameters registered with the optimizer.
- A trainable parameter may stay unchanged when its gradient is zero. Here the two prediction errors cancel in the bias gradient at the best bias-only solution.
- Restricting the trainable parameters can restrict which predictions the model can fit. Unfreezing the two weights lets these two examples have different predictions.
- Low training loss alone does not establish accuracy on unseen inputs. Use independent validation examples to assess that.

**Recorded experiments:** The first bias-only step changed bias `0` to about `0.3` while weights stayed `[1, 1]`, reducing loss from `10` to `8.29`. Repeated bias-only training approached bias `3`, predictions `[4, 4]`, and mean squared loss `1`, the minimum for these fixed weights and targets `[5, 3]`. After unfreezing and registering the weights with SGD, training reached loss about `0.00003164` and predictions `[4.9944, 3.0056]`.

These two training examples admit several parameter combinations. The learned weights near `[2, 0]` with bias `3` fit them, as do the original rule's weights `[3, 1]` with bias `2`. We have not established that the learned model recovered the original rule on other inputs.

## Vocabulary to revisit

| Term | Meaning in this experiment |
|---|---|
| Frozen parameter | A parameter kept fixed while training selected others |
| Trainable parameter | A parameter for which we calculate gradients and allow optimizer updates |
| `requires_grad_(False)` | Change an existing tensor so autograd does not accumulate gradients for it |
| Total parameter values | All scalar weight and bias values, frozen or trainable |
| Trainable parameter values | Scalar values in parameters whose `requires_grad` flag is true |

## Documentation

- [PyTorch: setting requires_grad](https://docs.pytorch.org/docs/stable/notes/autograd.html#setting-requires-grad)
- [PyTorch: Tensor.requires_grad_](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.requires_grad_.html)
